# Chapter 1 · Introduction to AI Agents

The same loop as the textbook, wired to a real model through Lumen, the University of Illinois campus LLM service. Every step is printed.

**Before you run:**

1. Sign in at [lumen.ncsa.illinois.edu](https://lumen.ncsa.illinois.edu/chat) with your Illinois account.
2. Open your [profile page](https://lumen.ncsa.illinois.edu/profile), scroll down to **API key**, and create one.
3. In Colab, add it as a secret named `LUMEN_API_KEY` (key icon in the left sidebar) and enable notebook access.

The notebook uses `glm-5.3-flash`. It is the recommended model for this workshop: fast, reliable with tool calls, and free on Lumen. Stick with it unless a section says otherwise.

In [ ]:
%pip install -q openai

In [ ]:
import os
from google.colab import userdata
os.environ["LUMEN_API_KEY"] = userdata.get("LUMEN_API_KEY")

from openai import OpenAI
client = OpenAI(
    base_url="https://lumen.ncsa.illinois.edu/v1",
    api_key=os.environ["LUMEN_API_KEY"],
)
MODEL = "glm-5.3-flash"  # highly preferred for this workshop

## Tools

The functions are tiny and read-only. The schemas are the contract the model sees. Note `strict: True` and `additionalProperties: False`. Lumen speaks the OpenAI chat-completions format, so each tool is wrapped as `{"type": "function", "function": {...}}` with the schema under `parameters`.

In [ ]:
FINANCIALS = {
    ("DE", "Q2-2026"):   {"name": "Deere", "revenue": 13.8e9, "yoy": 0.064},
    ("CAT", "Q2-2026"):  {"name": "Caterpillar", "revenue": 16.9e9, "yoy": 0.031},
    ("NVDA", "Q2-2026"): {"name": "NVIDIA", "revenue": 52.4e9, "yoy": 0.58},
}
PRICES = {"DE": 512.40, "CAT": 398.15, "NVDA": 181.22}

def get_financials(ticker, period="Q2-2026"):
    row = FINANCIALS.get((ticker, period))
    return {**row, "ticker": ticker, "period": period} if row else {"error": f"no data for {ticker} {period}"}

TOOLS = {"get_financials": get_financials}

TOOL_SCHEMAS = [
    {"type": "function",
     "function": {
         "name": "get_financials",
         "description": "Quarterly revenue and YoY growth for one ticker. Use only when the user asks about revenue, growth, or earnings.",
         "strict": True,
         "parameters": {"type": "object",
                        "properties": {"ticker": {"type": "string", "enum": ["DE", "CAT", "NVDA"]},
                                       "period": {"type": "string", "enum": ["Q2-2026"]}},
                        "required": ["ticker", "period"], "additionalProperties": False}}},
]

## The loop

Identical in shape to the textbook. The only differences are the message format the API expects (an assistant message carrying `tool_calls`, answered by one `tool` message per call) and that a reply can contain several tool calls at once, which we run and return together.

In [ ]:
import json, time

def _describe(name, args):
    """Plain-English description of a tool call, not a JSON blob — this book is for non-technical readers."""
    detail = ", ".join(str(v) for v in args.values())
    return f"{name.replace('_', ' ')}" + (f" ({detail})" if detail else "")

def _describe_result(result):
    """What a tool handed back, in words instead of a dict."""
    if isinstance(result, dict) and "error" in result:
        return f"nothing found — {result['error']}"
    if isinstance(result, dict):
        return ", ".join(f"{k.replace('_', ' ')} {v}" for k, v in result.items())
    return str(result)

def agent(question, tools=TOOLS, schemas=TOOL_SCHEMAS, system=None, max_steps=6):
    messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": question}]
    log = []
    for step in range(max_steps):
        response = client.chat.completions.create(model=MODEL, tools=schemas, messages=messages)
        msg = response.choices[0].message
        tool_calls = msg.tool_calls or []
        if not tool_calls:
            text = (msg.content or "").strip()
            log.append({"step": step, "kind": "text", "text": text})
            print(f"Step {step + 1}: answered — {text}")
            return text, log

        messages.append(msg)
        for tc in tool_calls:
            name, args = tc.function.name, json.loads(tc.function.arguments or "{}")
            t0 = time.time()
            try:
                result = tools[name](**args)
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}
            ms = round((time.time() - t0) * 1000, 2)
            log.append({"step": step, "kind": "tool_call", "tool": name, "args": args, "result": result, "ms": ms})
            print(f"Step {step + 1}: looked up {_describe(name, args)} -> {_describe_result(result)}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})

    log.append({"step": max_steps, "kind": "budget_exhausted"})
    return "Stopped: step budget exhausted.", log

answer, log = agent("What was Deere's revenue growth last quarter?")

## A live business example: trade pre-clearance

The same loop, now doing a job Champaign Capital Research pays two compliance officers to do. Two read-only tools: `get_trade_request` and `search_docs`. Deliberately no `clear_trade` tool: the model recommends, a compliance officer approves.

In [ ]:
TRADE_REQUESTS = {
    "7101": {"employee": "Priya Natarajan", "role": "senior analyst, industrials", "ticker": "DE",   "side": "buy",  "shares": 50,  "days_since_firm_research": 41, "restricted": False, "holding_days": None},
    "7102": {"employee": "Marcus Bell",     "role": "associate, industrials",     "ticker": "CAT",  "side": "sell", "shares": 200, "days_since_firm_research": 60, "restricted": False, "holding_days": 12},
    "7103": {"employee": "Jordan Lee",      "role": "analyst, semiconductors",    "ticker": "NVDA", "side": "buy",  "shares": 30,  "days_since_firm_research": 25, "restricted": True,  "holding_days": None},
    "7104": {"employee": "Tom Okafor",      "role": "data engineer",              "ticker": "DE",   "side": "sell", "shares": 80,  "days_since_firm_research": 6,  "restricted": False, "holding_days": 210},
}
DOCS = [
    {"id": "personal-trading-1", "text": "Employees must obtain compliance pre-clearance before trading any security in a sector the firm covers."},
    {"id": "personal-trading-2", "text": "Blackout window: no employee may trade a security within 14 days before or after the firm publishes research on it."},
    {"id": "personal-trading-3", "text": "Minimum holding period: positions in covered securities must be held at least 30 days before they are sold."},
    {"id": "restricted-list-1", "text": "Securities on the restricted list may not be traded by any employee; compliance maintains the list and reviews it weekly."},
    {"id": "research-process-1", "text": "Every figure in a published note must cite its source document or data vendor field; uncited figures block publication."},
]

def get_trade_request(request_id):
    row = TRADE_REQUESTS.get(request_id.lstrip("#"))
    return {**row, "request_id": request_id} if row else {"error": f"no request {request_id}"}

def search_docs(query, k=3):
    words = set(query.lower().split())
    scored = sorted(DOCS, key=lambda d: -sum(w in d["text"].lower() for w in words))
    return scored[:k]

PRECLEAR_TOOLS = {"get_trade_request": get_trade_request, "search_docs": search_docs}
PRECLEAR_SCHEMAS = [
    {"type": "function",
     "function": {
         "name": "get_trade_request", "strict": True,
         "description": "Look up one personal-trade pre-clearance request by id. Use when a question names a request number.",
         "parameters": {"type": "object", "properties": {"request_id": {"type": "string", "pattern": "^[0-9]{4}$"}},
                        "required": ["request_id"], "additionalProperties": False}}},
    {"type": "function",
     "function": {
         "name": "search_docs", "strict": True,
         "description": "Keyword search over the firm's policy handbook. Do not call with an empty or one-word query.",
         "parameters": {"type": "object", "properties": {"query": {"type": "string", "minLength": 4}, "k": {"type": "integer", "minimum": 1, "maximum": 5}},
                        "required": ["query", "k"], "additionalProperties": False}}},
]

SYSTEM = ("You are a research and compliance assistant at Champaign Capital Research, an equity research firm. "
          "For trade pre-clearance requests: look up the request, then the personal-trading policy, then give ONE recommendation "
          "in the form 'RECOMMEND: <APPROVE|HOLD|DECLINE>. <reason>' and cite policy chunk ids like [source: personal-trading-2]. "
          "You cannot clear trades; a compliance officer approves.")

In [ ]:
for rid in ["7101", "7102", "7103", "7104"]:
    answer, log = agent(f"Can compliance clear trade request #{rid}?", PRECLEAR_TOOLS, PRECLEAR_SCHEMAS, system=SYSTEM)
    print()

Compare with the textbook's mock run. Did the real model look up the request before the policy? Did every recommendation carry a `[source: …]` tag? If not, the fix belongs in the tool descriptions or the system prompt, and you should be able to say which.

## Exercise

1. Run the pre-clearance scenario above for all four requests and compare the tool sequence and recommendations with the textbook's mock run.
2. Add `get_price(ticker)` to `TOOLS` and a closed schema for it to `TOOL_SCHEMAS`, then run the question below. Expect four tool calls and one text answer.
3. Add the total elapsed time of the whole run to the final log entry.
4. Write three sentences: one tool call the model made that you would not have made, and the schema change that would prevent it.

In [ ]:
# 1. your get_price tool and schema here


# 3.
answer, log = agent("Is Deere's revenue growth better than Caterpillar's, and what are both trading at?")
print()
for entry in log:
    if entry["kind"] == "tool_call":
        print(f"Step {entry['step'] + 1}: looked up {_describe(entry['tool'], entry['args'])} ({entry['ms']} ms)")
    else:
        print(f"Step {entry['step'] + 1}: {entry.get('text', 'stopped')}")

## Optional: compare models

Lumen hosts more than one model; the list is in the model picker at [lumen.ncsa.illinois.edu/chat](https://lumen.ncsa.illinois.edu/chat). Set `MODEL` to a different one and rerun the exercise question. Count the tool calls and read the final answer. Which one would you ship, and why? (For everything else in this workshop, stay on `glm-5.3-flash`.)